In [ ]:
import polars as pl
from bs4 import BeautifulSoup
import re


In [38]:
data_x_train = pl.read_csv('/home/ubuntu/mar25_cmlops_rakuten/data/raw_data/X_train.csv')
X_train = data.slice(0, 1000)
data_y_train = pl.read_csv('/home/ubuntu/mar25_cmlops_rakuten/data/raw_data/Y_train.csv')

In [36]:
X_train.dtypes

[Int64, String, String, Int64, Int64]

In [ ]:
def clean_description(text: str) -> str:
    
    # Nettoyage HTML avec BeautifulSoup
    soup = BeautifulSoup(text, "html.parser")
    cleaned_text = soup.get_text(separator="\n", strip=True)
    
    # Transformer les retours à la ligne en phrases continues
    cleaned_text = cleaned_text.replace("\n", ". ")
    # Remplacer les entités HTML comme &#39; ou autres
    cleaned_text = re.sub(r'&#\d+;', '', cleaned_text)  

     # Remplacer les caractères non désirés comme '¿¿' par des espaces
    cleaned_text = re.sub(r'¿{2,}', ' ', cleaned_text)
    cleaned_text = cleaned_text.replace('¿', '')
    # Supprimer tous les autres caractères spéciaux non désirés (tels que ®, ©, ™, etc.)
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', cleaned_text)  # Enlever les caractères non-ASCII
    return cleaned_text

# Application de la fonction sur la colonne 'description' du DataFrame Polars
data_x_train = data_x_train.with_columns(
    [
        pl.col("description").map_elements(lambda text:clean_description(text), return_dtype=pl.Utf8),
        pl.col("designation").map_elements(clean_description, return_dtype=pl.Utf8)
)
X_train_cleaned.write_csv("/home/ubuntu/mar25_cmlops_rakuten/data/explore_data/X_train.csv", separator=",")

/tmp/ipykernel_1570/4253998028.py:4: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(text, "html.parser")


In [48]:
comptage_classe = data_y_train.group_by("prdtypecode").agg(pl.count().alias("nb_occurrences")).sort("nb_occurrences",descending =True)
comptage_classe.write_csv("/home/ubuntu/mar25_cmlops_rakuten/data/explore_data/comptage_classe.csv", separator=",")

/tmp/ipykernel_1570/2847208420.py:1: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  comptage_classe = data_y_train.group_by("prdtypecode").agg(pl.count().alias("nb_occurrences")).sort("nb_occurrences",descending =True)
